# 04 — SQL Analytics: WO-1 Readiness Audit

## Purpose

WO-1 establishes the verified baseline for analytical modelling. It reconciles the existing quality registry and STAGING row counts, inspects current database structures, validates student-presentation keys, and measures assessment-join fan-out.

Canonical SQL definitions live in `sql/04_analytics/`. This notebook invokes them by query ID through `src/analytics/db.py`; SQL is not duplicated here.

## Execution controls

- Each query uses a separate PostgreSQL connection with read-only session settings, a 120-second statement timeout, and a 5-second lock timeout.
- Displayed results are capped at 500 rows. Exact counts are computed in PostgreSQL; the full `student_vle` table is not loaded into Pandas.
- RAW loading, STAGING creation, quality-registry seeding, and DDL are outside this work order.
- AN2.01 and AN2.02 depend on successful key validation in AN1.01.
- Historical counts are reference values, not evidence of the current database state.

## Work-order status

**Status:** The saved WO-1 execution evidence has been reviewed, and the targeted category reconciliation for AN2.02, AN2.04, and AN2.05 is complete. All 132 presentation/dimension groups passed category-count and rounded-percentage reconciliation, with one consistent denominator per group. This notebook preserves historical execution evidence; no SQL was rerun during the documentation correction.

## 0. Query helper and connection controls

The helper resolves canonical SQL files by stable query ID, opens a separate read-only database session for each query, and returns a bounded result table with execution time. Connection settings are read from the local environment and are never displayed.

In [1]:
# 1. Load the project root and canonical query runner.
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "sql" / "04_analytics").exists():
    PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "sql" / "04_analytics").exists():
    raise FileNotFoundError("Open this notebook from the repository or notebooks folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analytics.db import run_query

query_results = {}
print("Project root:", PROJECT_ROOT.name)
print("Canonical SQL folder found:", (PROJECT_ROOT / "sql" / "04_analytics").exists())

Project root: OULAD-EdTech-Learning-Analytics
Canonical SQL folder found: True


In [2]:
print("run_query defined:", "run_query" in globals())
print("query_results defined:", "query_results" in globals())
print("Existing results in memory:", len(query_results))

run_query defined: True
query_results defined: True
Existing results in memory: 0


**Validation status:** The import and path checks establish whether the canonical SQL directory and query helper are available. No database result is produced by this cell.

## AN0.01 — Database identity and session safety

**Question:** Is the active connection using the intended database and read-only session settings?

**Expectation:** The database is `oulad_analytics`; read-only mode is enabled.

**Interpretation:** A database or session-setting mismatch invalidates subsequent results and must be resolved before the audit proceeds.

In [2]:
# 1. Verify the active database and read-only session before continuing.
result = run_query("AN0.01")
query_results["AN0.01"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

row = result["data"].iloc[0]
if row["database_name"] != "oulad_analytics" or str(row["transaction_read_only"]).lower() != "on":
    raise RuntimeError("Safety gate failed. Stop the notebook and investigate the active connection.")

,database_name,database_user,current_schema,server_version,transaction_read_only,default_transaction_read_only,statement_timeout,lock_timeout
0,oulad_analytics,postgres,public,18.6,on,on,2min,5s


Query file: sql\04_analytics\AN0.01_database_identity.sql
Elapsed seconds: 0.026
Displayed rows: 1; truncated: False


### Interpretation and decision — AN0.01

**Observed result:** The saved output identifies database `oulad_analytics`, PostgreSQL 18.6, and confirms both `transaction_read_only` and `default_transaction_read_only` are `on`. The statement timeout is 2 minutes and the lock timeout is 5 seconds.

**Decision:** The recorded session satisfies the intended database identity and read-only session safeguards for these diagnostics.

**Limitations:** These are saved notebook results; they do not prove the state of a new connection opened later. Credentials are not displayed.

## AN0.02 — Existing quality registry snapshot

**Question:** Does the live registry match the documented historical record?

**Historical reference:** 29 records and 29 distinct test codes; 25 `PASS`, 4 `REVIEW`, and 0 `FAIL`. Historical `executed_at` values may be NULL.

**Interpretation:** A difference is a reconciliation issue. This query reads the registry; it does not rerun or reseed completed engineering checks.

In [3]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN0.02")
query_results["AN0.02"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,metric_name,metric_value
0,distinct_test_codes,29
1,duplicate_test_code_excess,0
2,fail_rows,0
3,null_executed_at_rows,29
4,pass_rows,25
5,registry_rows,29
6,review_rows,4


Query file: sql\04_analytics\AN0.02_quality_registry_snapshot.sql
Elapsed seconds: 0.098
Displayed rows: 7; truncated: False


### Interpretation and decision — AN0.02

**Observed result:** The live-registry snapshot recorded in the notebook has 29 rows and 29 distinct test codes, zero duplicate-code excess, 25 PASS, 4 REVIEW, zero FAIL, and 29 NULL `executed_at` values.

**Decision:** Preserve the four REVIEW findings for explicit disposition. Do not fabricate historical execution timestamps or rerun seed operations just to populate them.

**Limitations:** A registry status is not a substitute for examining the evidence behind each REVIEW case. The result is saved notebook evidence, not a fresh execution in this session.

## AN0.03 — STAGING row-count reconciliation

**Question:** Do current STAGING row counts match the established baseline?

**Historical reference:** `courses` 22; `assessments` 206; `student_info` 32,593; `student_registration` 32,593; `student_assessment` 173,912; `vle` 6,364; `student_vle` 10,655,280.

**Interpretation:** PostgreSQL computes exact counts and returns seven summary rows. The large `student_vle` scan is subject to the configured timeout; no table data is transferred to Pandas.

In [4]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN0.03")
query_results["AN0.03"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,table_name,expected_rows,actual_rows,row_count_difference,reconciliation_status
0,assessments,206,206,0,MATCH
1,courses,22,22,0,MATCH
2,student_assessment,173912,173912,0,MATCH
3,student_info,32593,32593,0,MATCH
4,student_registration,32593,32593,0,MATCH
5,student_vle,10655280,10655280,0,MATCH
6,vle,6364,6364,0,MATCH


Query file: sql\04_analytics\AN0.03_staging_row_count_reconciliation.sql
Elapsed seconds: 3.651
Displayed rows: 7; truncated: False


### Interpretation and decision — AN0.03

**Observed result:** All seven STAGING row counts match their documented reference values: courses 22; assessments 206; student_info 32,593; student_registration 32,593; student_assessment 173,912; vle 6,364; and student_vle 10,655,280.

**Decision:** Accept row-count reconciliation as evidence that the documented row-count baseline matches the saved snapshot. It does not establish semantic correctness, key uniqueness, or absence of repeated activity records.

**Limitations:** Counts should be rechecked only if a later task needs current database-state confirmation or if the underlying data-loading state changes.

## AN0.04 — STAGING sizes and indexes

**Question:** What physical relation sizes and indexes exist in the current database?

**Expectation:** The live catalog, not assumptions about CTAS, determines which indexes exist.

**Interpretation:** Record relation sizes and existing index definitions as a performance baseline. This query creates no indexes or other database objects.

In [5]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN0.04")
query_results["AN0.04"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,schema_name,table_name,table_size,total_size_including_indexes,index_count,index_names
0,staging,student_vle,612 MB,612 MB,0,(none)
1,staging,student_assessment,9216 kB,9256 kB,0,(none)
2,staging,student_info,3720 kB,3760 kB,0,(none)
3,staging,student_registration,1664 kB,1704 kB,0,(none)
4,staging,vle,336 kB,376 kB,0,(none)
5,staging,assessments,16 kB,24 kB,0,(none)
6,staging,courses,8192 bytes,16 kB,0,(none)


Query file: sql\04_analytics\AN0.04_staging_relation_sizes_and_indexes.sql
Elapsed seconds: 0.304
Displayed rows: 7; truncated: False


### Interpretation and decision — AN0.04

**Observed result:** The saved catalog snapshot shows no indexes on the seven STAGING tables. `student_vle` occupies approximately 612 MB; `student_assessment` is approximately 9 MB, and the other tables are smaller.

**Decision:** Record the measured catalog state as a performance consideration for future queries. Do not create indexes as part of this diagnostic; any persistent index change requires a separate cost-benefit review and explicit approval.

**Limitations:** Table sizes and index inventory are a snapshot, not a benchmark of query performance. The output does not itself justify a database modification.

## AN1.01 — Student key completeness and uniqueness

**Question:** Is `(code_module, code_presentation, id_student)` complete and unique in both `student_info` and `student_registration`?

**Expectation:** No key components are NULL and no duplicate key groups exist.

**Interpretation:** AN2 profile queries are valid only if both tables pass this key check.

In [6]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN1.01")
query_results["AN1.01"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,table_name,total_rows,null_key_rows,distinct_key_values,duplicate_key_groups,excess_rows,gate_status
0,student_info,32593,0,32593,0,0,PASS
1,student_registration,32593,0,32593,0,0,PASS


Query file: sql\04_analytics\AN1.01_student_key_uniqueness.sql
Elapsed seconds: 0.316
Displayed rows: 2; truncated: False


### Interpretation and decision — AN1.01

**Observed result:** Both `student_info` and `student_registration` contain 32,593 rows, zero rows with NULL key components, 32,593 distinct composite keys, zero duplicate-key groups, and zero excess rows. Both gate statuses are PASS.

**Decision:** The tested composite key `(code_module, code_presentation, id_student)` is suitable for identifying student-presentation rows in these two tables. AN1.02 can compare their key sets.

**Limitations:** This key validation does not imply that `id_student` is unique across all presentations or that the `student_vle` activity key is unique.

## AN1.02 — Student key-set reconciliation

**Question:** Do `student_info` and `student_registration` contain the same composite-key population, and how do their distinct student counts compare?

**Expectation:** No keys are exclusive to either table if both represent the same student-presentation population.

**Interpretation:** Read this result alongside AN1.01; equal row counts alone do not prove that the key sets match.

In [7]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN1.02")
query_results["AN1.02"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,metric_name,metric_value
0,shared_complete_keys,32593
1,student_info_distinct_student_ids,28785
2,student_info_only_complete_keys,0
3,student_registration_distinct_student_ids,28785
4,student_registration_only_complete_keys,0


Query file: sql\04_analytics\AN1.02_student_key_set_reconciliation.sql
Elapsed seconds: 0.252
Displayed rows: 5; truncated: False


### Interpretation and decision — AN1.02

**Observed result:** The two student tables share 32,593 complete composite keys, with zero keys exclusive to either table. Each table contains 28,785 distinct student IDs.

**Decision:** The tested student-presentation populations reconcile across the two tables. The difference between 32,593 presentation rows and 28,785 distinct people represents repeated appearances across presentations, not missing people.

**Limitations:** This reconciliation validates the tested key sets, not every attribute value or temporal relationship in the tables.

## AN1.03 — Assessment join fan-out

**Question:** How much does joining `student_assessment` to `student_info` by `id_student` alone inflate the row count compared with the assessment-metadata path?

**Expectation:** The student-only join may multiply rows because one student can appear in multiple module presentations. The defensible route joins through `assessments` and matches the module and presentation.

**Interpretation:** A multiplier above 1 indicates join fan-out, not additional assessment records. Unmatched records must also be considered.

In [8]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN1.03")
query_results["AN1.03"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,source_rows,naive_join_rows,correct_join_rows,naive_join_multiplier,correct_join_multiplier,unmatched_assessment_rows,unmatched_student_presentation_rows,matched_student_presentation_rows
0,173912,207319,173912,1.1921,1.0000,0,0,173912


Query file: sql\04_analytics\AN1.03_student_assessment_join_fanout.sql
Elapsed seconds: 0.296
Displayed rows: 1; truncated: False


### Interpretation and decision — AN1.03

**Observed result:** The source has 173,912 student-assessment rows. Joining to student profiles by `id_student` alone produces 207,319 rows, a multiplier of 1.1921. The metadata-aware join preserves 173,912 rows, has a multiplier of 1.0000, and reports zero unmatched assessment rows or unmatched student-presentation rows.

**Decision:** Use the assessment metadata path and match on the relevant module, presentation, and student context. Do not use the student-only join for outcome metrics because it creates row multiplication.

**Limitations:** This validates the tested join paths and counts; future joins should still have their grain and cardinality checked explicitly.

## AN1.04 — Course-presentation relationship validation

**Question:** Do student-presentation rows map to exactly one course metadata row?

**Expectation:** No duplicate course keys, no unmatched student-presentation rows, and a join multiplier of 1.0000.

**Grain:** One row per module-presentation relationship check.

**Source:** student_info and courses

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Use this relationship only if the measured key and join checks pass.

In [4]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN1.04")
query_results["AN1.04"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,diagnostic,table_name,source_rows,distinct_course_keys,matched_join_rows,unmatched_rows,duplicate_key_groups,duplicate_excess_rows,joined_rows,join_multiplier
0,COURSE_KEY,courses,22,22.0,NaN,NaN,0.0,0.0,NaN,None
1,STUDENT_COURSE_JOIN,student_info,32593,NaN,32593.0,0.0,NaN,NaN,32593.0,1.0000
2,STUDENT_COURSE_JOIN,student_registration,32593,NaN,32593.0,0.0,NaN,NaN,32593.0,1.0000


Query file: sql\04_analytics\AN1.04_course_presentation_relationships.sql
Elapsed seconds: 0.315
Displayed rows: 3; truncated: False


### Interpretation and decision — AN1.04

**Observed result:** The `courses` table contains 22 rows and 22 distinct course-presentation keys, with no duplicate-key groups or excess rows. Joining either `student_info` or `student_registration` to course metadata preserves all 32,593 source rows, produces zero unmatched rows, and has a join multiplier of 1.0000.

**Decision:** The course-presentation metadata join is suitable for the current presentation-level analyses. This validates the course key and join cardinality; it does not validate the repeated composite key in `student_vle`.

**Limitations:** This is saved notebook evidence, not a fresh execution in this session.

## AN1.05 — Assessment metadata relationship validation

**Question:** Does each student_assessment row map to one assessment metadata row through the assessment identifier?

**Expectation:** No duplicate assessment identifiers, no unmatched submissions, and a join multiplier of 1.0000.

**Grain:** Assessment relationship and submission rows.

**Source:** student_assessment and assessments

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Use the metadata join path only after the reported cardinality checks pass.

In [5]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN1.05")
query_results["AN1.05"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,diagnostic,source_rows,distinct_assessment_ids,matched_join_rows,unmatched_rows,duplicate_key_groups,duplicate_excess_rows,joined_rows,join_multiplier
0,ASSESSMENT_KEY,206,206.0,NaN,NaN,0.0,0.0,NaN,None
1,SUBMISSION_JOIN,173912,NaN,173912.0,0.0,NaN,NaN,173912.0,1.0000


Query file: sql\04_analytics\AN1.05_assessment_metadata_relationships.sql
Elapsed seconds: 0.334
Displayed rows: 2; truncated: False


### Interpretation and decision — AN1.05

**Observed result:** All 206 assessment metadata rows have distinct assessment IDs, with no duplicate assessment-key groups. Joining student assessment records through assessment metadata preserves all 173,912 source rows, with zero unmatched rows and a 1.0000 join multiplier.

**Decision:** Retain the assessment-metadata join path. Do not join assessment submissions to student profiles by `id_student` alone; AN1.03 showed that this inflates the row count.

**Limitations:** These results validate the tested identifier and join path, not every assessment business rule. This is saved notebook evidence, not a fresh execution in this session.

## AN1.06 — Registration grain versus distinct people

**Question:** How do student-presentation registration counts differ from distinct people?

**Expectation:** Registration rows should be greater than or equal to distinct people; presentation participation per person should be explicit.

**Grain:** Registration rows and distinct student IDs at the reported grouping.

**Source:** student_registration

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Do not treat registration records as unique people across presentations.

In [6]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN1.06")
query_results["AN1.06"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,metric_group,presentation_count,registration_rows,people_count,null_registration_date_rows,rows_with_unregistration_date
0,OVERALL,NaN,32593.0,28785,45.0,10072.0
1,PEOPLE_BY_PRESENTATION_COUNT,1.0,NaN,25247,NaN,NaN
2,PEOPLE_BY_PRESENTATION_COUNT,2.0,NaN,3293,NaN,NaN
3,PEOPLE_BY_PRESENTATION_COUNT,3.0,NaN,221,NaN,NaN
4,PEOPLE_BY_PRESENTATION_COUNT,4.0,NaN,23,NaN,NaN
5,PEOPLE_BY_PRESENTATION_COUNT,5.0,NaN,1,NaN,NaN


Query file: sql\04_analytics\AN1.06_registration_person_grain.sql
Elapsed seconds: 0.293
Displayed rows: 6; truncated: False


### Interpretation and decision — AN1.06

**Observed result:** The registration table contains 32,593 student-presentation rows but 28,785 distinct student IDs. The presentation-count distribution is: one presentation, 25,247 people; two, 3,293; three, 221; four, 23; and five, 1. There are 45 NULL registration dates and 10,072 rows with a non-NULL unregistration date.

**Decision:** Use the student-presentation composite key for presentation-level analysis and distinguish registration rows from distinct people in denominators. Do not automatically classify NULL dates as errors or interpret an unregistration date as a confirmed withdrawal outcome.

**Limitations:** This descriptive profile does not explain why dates are missing or why students appear in multiple presentations.

## AN1.07 — Activity coverage by presentation

**Question:** How much VLE activity is recorded in each registered presentation?

**Expectation:** One row per registered presentation, including registered students, active students, active days, sites, negative-day rows, and stored click sum.

**Grain:** Module-presentation aggregate.

**Source:** student_registration and student_vle

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Allow up to 600,000 ms for this activity-scale query; execute separately and review the returned timing.

**Read as:** Use the registered population as the denominator and preserve the recorded activity grain.

In [7]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN1.07", statement_timeout_ms=600000)
query_results["AN1.07"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,registration_rows,registered_students,activity_rows,students_with_any_activity,matched_active_students,activity_students_not_registered,pct_registered_students_with_matched_activity,distinct_sites_with_activity,distinct_activity_days,negative_day_rows,minimum_activity_day,maximum_activity_day,stored_sum_click
0,AAA,2013J,383,383,180982,378,378,0,98.69,209,279,10847,-10,268,648494
1,AAA,2014J,365,365,169316,357,357,0,97.81,197,294,14508,-24,269,598158
2,BBB,2013B,1767,1767,403266,1537,1537,0,86.98,315,250,19738,-9,240,1347911
3,BBB,2013J,2237,2237,452638,1870,1870,0,83.59,320,280,19826,-23,268,1378656
4,BBB,2014B,1613,1613,273236,1294,1294,0,80.22,311,242,11900,-9,234,833865
5,BBB,2014J,2292,2292,438424,1921,1921,0,83.81,206,272,27058,-9,262,1673744
6,CCC,2014B,1936,1936,496181,1681,1681,0,86.83,180,258,26648,-18,241,1889177
7,CCC,2014J,2498,2498,711646,2302,2302,0,92.15,220,288,42604,-18,269,2792972
8,DDD,2013B,1303,1303,536837,1214,1214,0,93.17,408,257,44913,-16,240,1387467
9,DDD,2013J,1938,1938,680806,1768,1768,0,91.23,456,280,59515,-18,261,1757214


Query file: sql\04_analytics\AN1.07_activity_coverage_by_presentation.sql
Elapsed seconds: 47.94
Displayed rows: 22; truncated: False


### Interpretation and decision — AN1.07

**Observed result:** All activity-bearing students in the saved output match a registered student within the same module and presentation; no activity-bearing students fall outside the registered population. Activity coverage ranges from 80.22% in BBB 2014B to 98.69% in AAA 2013J. Negative activity-day values occur in every presentation.

**Decision:** Use registered students in the same presentation as the explicit denominator. Preserve negative day values for investigation rather than deleting or silently recoding them. Interpret `activity_rows` as source activity records and `stored_sum_click` as the sum of stored click counts—not deduplicated activity.

**Limitations:** Missing activity does not prove no learning or withdrawal. Repeated `student_vle` keys remain unresolved, so click-based metrics require a separately documented grain/handling decision. This is saved notebook evidence, not a fresh execution.

## AN1.08 — Registrations without activity

**Question:** Which registered students have no matching VLE activity rows in their presentation?

**Expectation:** Counts and rates should be reported at module-presentation grain using registrations as the denominator.

**Grain:** Registered student-presentation keys and matched activity keys.

**Source:** student_registration and student_vle

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Allow up to 600,000 ms for this activity-scale query; execute separately and review the returned timing.

**Read as:** Interpret rates only with the registration denominator and exact composite key.

In [8]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN1.08", statement_timeout_ms=600000)
query_results["AN1.08"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,registration_rows,registrations_with_activity,registrations_without_activity,pct_registrations_without_activity
0,AAA,2013J,383,378,5,1.31
1,AAA,2014J,365,357,8,2.19
2,BBB,2013B,1767,1537,230,13.02
3,BBB,2013J,2237,1870,367,16.41
4,BBB,2014B,1613,1294,319,19.78
5,BBB,2014J,2292,1921,371,16.19
6,CCC,2014B,1936,1681,255,13.17
7,CCC,2014J,2498,2302,196,7.85
8,DDD,2013B,1303,1214,89,6.83
9,DDD,2013J,1938,1768,170,8.77


Query file: sql\04_analytics\AN1.08_registrations_without_activity.sql
Elapsed seconds: 2.961
Displayed rows: 22; truncated: False


### Interpretation and decision — AN1.08

**Observed result:** Registration rows without matching VLE activity reconcile with total registration rows by presentation. The largest count is 371 in BBB 2014J, while the largest percentage is 19.78% (319 rows) in BBB 2014B.

**Decision:** Report both counts and percentages, naming the registration-row denominator. Use “no recorded matching activity”; do not relabel these records as withdrawals, disengagement, or lack of learning.

**Limitations:** This checks for at least one matching activity row at student-presentation grain. It does not assess activity intensity or resolve repeated `student_vle` keys. This is saved notebook evidence, not a fresh execution.

## AN1.09 — VLE resource match coverage

**Question:** How many recorded activity rows match VLE resource metadata?

**Expectation:** Matched and unmatched activity rows should reconcile to the activity-row denominator.

**Grain:** Activity rows and resource metadata matches.

**Source:** student_vle and vle

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Allow up to 600,000 ms for this activity-scale query; execute separately and review the returned timing.

**Read as:** Report row-level match coverage and avoid treating unmatched records as zero activity.

In [9]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN1.09", statement_timeout_ms=600000)
query_results["AN1.09"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,diagnostic,match_status,row_count,distinct_site_ids,duplicate_groups,duplicate_excess_rows
0,ACTIVITY_RESOURCE_MATCH,EXACT_PRESENTATION_MATCH,10655280,NaN,NaN,NaN
1,VLE_KEY,NaN,6364,6364.0,0.0,0.0


Query file: sql\04_analytics\AN1.09_vle_resource_match_coverage.sql
Elapsed seconds: 17.263
Displayed rows: 2; truncated: False


### Interpretation and decision — AN1.09

**Observed result:** All 10,655,280 `student_vle` activity rows have an exact presentation/resource metadata match. The `vle` table has 6,364 rows and 6,364 distinct site IDs, with no duplicate site-ID groups or excess rows.

**Decision:** Resource metadata coverage is complete for the tested join. Keep metadata matching separate from activity-row uniqueness: an exact match to `vle` does not make the `student_vle` composite key unique.

**Limitations:** This does not resolve repeated activity keys, differing `sum_click` values within repeated groups, or the appropriate metric grain. This is saved notebook evidence, not a fresh execution.

## AN2.01 — Student counts by presentation

**Question:** Do `student_info` and `student_registration` counts reconcile at module-presentation grain?

**Expectation:** Counts should reconcile when both tables represent the same validated key population.

**Execution gate:** This query runs only when AN1.01 passes for both tables.

In [9]:
# 1. Run this profile only when both student tables passed AN1.01.
gate = query_results.get("AN1.01")
if gate is None or len(gate["data"]) != 2 or not (gate["data"]["gate_status"] == "PASS").all():
    print("SKIPPED: AN1.01 did not confirm complete, unique keys in both student tables.")
else:
    result = run_query("AN2.01")
    query_results["AN2.01"] = result
    display(result["data"])
    print(f"Query file: {result['query_file']}")
    print(f"Elapsed seconds: {result['elapsed_seconds']}")
    print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,student_info_rows,registration_rows,student_info_distinct_students,registration_distinct_students,row_count_difference
0,AAA,2013J,383,383,383,383,0
1,AAA,2014J,365,365,365,365,0
2,BBB,2013B,1767,1767,1767,1767,0
3,BBB,2013J,2237,2237,2237,2237,0
4,BBB,2014B,1613,1613,1613,1613,0
5,BBB,2014J,2292,2292,2292,2292,0
6,CCC,2014B,1936,1936,1936,1936,0
7,CCC,2014J,2498,2498,2498,2498,0
8,DDD,2013B,1303,1303,1303,1303,0
9,DDD,2013J,1938,1938,1938,1938,0


Query file: sql\04_analytics\AN2.01_student_presentation_profile.sql
Elapsed seconds: 0.036
Displayed rows: 22; truncated: False


### Interpretation and decision — AN2.01

**Observed result:** The saved profile contains 22 module-presentation rows. For every presentation, `student_info` and `student_registration` counts agree, and each row count equals the distinct-student count. Population ranges from 365 in AAA 2014J to 2,498 in CCC 2014J.

**Decision:** Use the module-presentation population as the denominator for presentation-level descriptive metrics, while naming the denominator in each result.

**Limitations:** These counts represent student-presentation records, not unique people across the entire dataset. The result is a saved notebook snapshot, not a fresh execution.

## AN2.02 — Final-result distribution

**Question:** What final-result categories are observed within each module presentation, including missing outcomes?

**Expectation:** Within-presentation percentages sum to approximately 100%.

**Denominator:** All rows in `student_info` within the relevant module presentation, including missing `final_result` values.

**Interpretation:** This is a descriptive distribution; it does not establish causes of success, failure, or withdrawal.

In [7]:
# 1. Run this profile only when both student tables passed AN1.01.
gate = query_results.get("AN1.01")
if gate is None or len(gate["data"]) != 2 or not (gate["data"]["gate_status"] == "PASS").all():
    print("SKIPPED: AN1.01 did not confirm complete, unique keys in both student tables.")
else:
    result = run_query("AN2.02")
    query_results["AN2.02"] = result
    display(result["data"])
    print(f"Query file: {result['query_file']}")
    print(f"Elapsed seconds: {result['elapsed_seconds']}")
    print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,final_result_category,student_rows,presentation_student_rows,percentage_within_presentation
0,AAA,2013J,Distinction,20,383,5.22
1,AAA,2013J,Fail,45,383,11.75
2,AAA,2013J,Pass,258,383,67.36
3,AAA,2013J,Withdrawn,60,383,15.67
4,AAA,2014J,Distinction,24,365,6.58
...,...,...,...,...,...,...
83,GGG,2014B,Withdrawn,100,833,12.00
84,GGG,2014J,Distinction,127,749,16.96
85,GGG,2014J,Fail,179,749,23.90
86,GGG,2014J,Pass,317,749,42.32


Query file: sql\04_analytics\AN2.02_final_result_distribution.sql
Elapsed seconds: 0.047
Displayed rows: 88; truncated: False


### Interpretation and decision — AN2.02

**Observed result:** The saved output contains 88 rows, with four recorded final-result categories represented across 22 module presentations. Counts and percentages are expressed within each presentation, and the displayed categories reconcile to their presentation populations.

**Decision:** Retain the within-presentation denominator and keep outcome categories explicit. Before publishing category-specific claims, inspect the full 88-row table and verify whether any missing-result category is present rather than assuming it.

**Limitations:** These are descriptive outcome distributions. They do not establish causal effects or explain why a student received a particular result. This is saved notebook evidence, not a fresh execution.

**Output completeness note:** The query runner reports `truncated: False`, but the rendered Pandas representation abbreviates middle rows. Verify category totals and missing-outcome handling with a compact reconciliation output before publishing category-specific comparisons.

## WO-1 reconciliation — AN2.02: Final-result categories

**Question:** Do category counts reconcile exactly to the relevant presentation denominator, retaining explicit NULL categories?

**Expectation:** Counts equal the denominator; the sum of rounded category percentages is approximately 100%; one denominator value exists per group.

**Grain:** One row per module presentation.

**Source:** Existing canonical read-only query `AN2.02` over `staging.student_info`.

**Safety:** READ-ONLY. No source data modification, imputation, or category collapsing.

**Runtime:** Low.

**Read as:** This verifies aggregation completeness and denominator consistency, not causal relationships. 88 rows; explicit <NULL> category is retained when present.


In [10]:
# 1. Run the existing canonical read-only query and reconcile category totals.
recon = run_query("AN2.02")
df = recon["data"].copy()
required = {"code_module", "code_presentation", "final_result_category", "student_rows", "presentation_student_rows", "percentage_within_presentation"}
assert required.issubset(df.columns), f"Unexpected columns: {list(df.columns)}"
assert not recon["display_truncated"], "Result was truncated by the query runner."

group_cols = ["code_module", "code_presentation"]
check = (
    df.groupby(group_cols, dropna=False)
    .agg(
        category_total=("student_rows", "sum"),
        denominator=("presentation_student_rows", "first"),
        denominator_values=("presentation_student_rows", "nunique"),
        percentage_total=("percentage_within_presentation", "sum"),
        null_category_present=("final_result_category", lambda s: s.eq("<NULL>").any()),
        categories=("final_result_category", "nunique"),
    )
    .reset_index()
)
check["counts_reconcile"] = check["category_total"].eq(check["denominator"]) & check["denominator_values"].eq(1)
check["percentages_reconcile"] = check["percentage_total"].between(99.90, 100.10)
display(check)
print(f"Query: {recon['query_file']} | elapsed: {recon['elapsed_seconds']} sec | rows: {recon['rows_displayed']}")
print("All counts reconcile:", bool(check["counts_reconcile"].all()))
print("All rounded percentages reconcile:", bool(check["percentages_reconcile"].all()))
print("Groups with explicit <NULL> category:", int(check["null_category_present"].sum()), "of", len(check))


,code_module,code_presentation,category_total,denominator,denominator_values,percentage_total,null_category_present,categories,counts_reconcile,percentages_reconcile
0,AAA,2013J,383,383,1,100.00,False,4,True,True
1,AAA,2014J,365,365,1,100.00,False,4,True,True
2,BBB,2013B,1767,1767,1,100.00,False,4,True,True
3,BBB,2013J,2237,2237,1,100.00,False,4,True,True
4,BBB,2014B,1613,1613,1,100.00,False,4,True,True
5,BBB,2014J,2292,2292,1,100.00,False,4,True,True
6,CCC,2014B,1936,1936,1,100.00,False,4,True,True
7,CCC,2014J,2498,2498,1,99.99,False,4,True,True
8,DDD,2013B,1303,1303,1,100.00,False,4,True,True
9,DDD,2013J,1938,1938,1,100.00,False,4,True,True


Query: sql\04_analytics\AN2.02_final_result_distribution.sql | elapsed: 0.078 sec | rows: 88
All counts reconcile: True
All rounded percentages reconcile: True
Groups with explicit <NULL> category: 0 of 22



### Interpretation and decision — AN2.02 category reconciliation

**Execution status:** Executed and validated.

**Observed result:**
- Reconciliation grain: one row per module-presentation.
- Presentation groups checked: 22.
- Category counts reconciled with the presentation denominator: 22/22.
- Rounded percentage totals reconciled within the accepted tolerance: 22/22.
- Groups with an explicit `<NULL>` output category: 0/22.
- Underlying SQL output: 88 rows.
- Elapsed time: 0.078 seconds.

**Validation:**
Every presentation had exactly one denominator value. The sum of category counts equalled the corresponding presentation population. Rounded percentage totals were approximately 100%.

**Interpretation:**
The recorded final-result distribution is internally consistent at module-presentation grain. The reconciliation validates aggregation completeness and denominator consistency; it does not establish causal relationships or explain missing values.

**NULL limitation:**
No explicit `<NULL>` category was emitted by this grouped output. This does not prove that the source column contains no SQL NULL values.

**Decision:**
Accept AN2.02 category-count and percentage reconciliation for descriptive reporting, subject to the documented interpretation limits.


## AN2.03 — Presentation size profile

**Question:** How does the student population vary across module presentations?

**Expectation:** One row per module-presentation with counts and rates derived from the validated course-presentation relationship.

**Grain:** Module-presentation aggregate.

**Source:** student_info and courses

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Use counts for scale comparisons; do not interpret size as a causal factor.

In [10]:
# 1. Run this profile only after AN1.04 confirms unique course keys and complete, one-to-one student-course matches.
gate = query_results.get("AN1.04")
if gate is None or gate["data"].empty:
    print("SKIPPED: AN1.04 has not returned course-presentation diagnostics.")
else:
    data = gate["data"]
    course_key = data.loc[data["diagnostic"] == "COURSE_KEY"]
    joins = data.loc[data["diagnostic"] == "STUDENT_COURSE_JOIN"]
    gate_passes = (
        len(course_key) == 1
        and len(joins) == 2
        and int(course_key.iloc[0]["duplicate_key_groups"]) == 0
        and (joins["unmatched_rows"] == 0).all()
        and (joins["join_multiplier"] == 1.0).all()
    )
    if not gate_passes:
        print("SKIPPED: AN1.04 did not confirm unique course keys and complete one-to-one joins.")
        display(data)
    else:
        result = run_query("AN2.03")
        query_results["AN2.03"] = result
        display(result["data"])
        print(f"Query file: {result['query_file']}")
        print(f"Elapsed seconds: {result['elapsed_seconds']}")
        print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,module_presentation_length,student_presentation_rows,distinct_students_in_presentation,within_presentation_key_status
0,AAA,2013J,268,383,383,MATCH
1,AAA,2014J,269,365,365,MATCH
2,BBB,2013B,240,1767,1767,MATCH
3,BBB,2013J,268,2237,2237,MATCH
4,BBB,2014B,234,1613,1613,MATCH
5,BBB,2014J,262,2292,2292,MATCH
6,CCC,2014B,241,1936,1936,MATCH
7,CCC,2014J,269,2498,2498,MATCH
8,DDD,2013B,240,1303,1303,MATCH
9,DDD,2013J,261,1938,1938,MATCH


Query file: sql\04_analytics\AN2.03_presentation_size_profile.sql
Elapsed seconds: 0.06
Displayed rows: 22; truncated: False


### Interpretation and decision — AN2.03

**Observed result:** The profile returns 22 module-presentation rows. In every presentation, the student-presentation row count equals the distinct student count and the key-status field is `MATCH`. Population ranges from 365 students in AAA 2014J to 2,498 in CCC 2014J.

**Decision:** Use this as a presentation-level population reference for subsequent descriptive profiles. State the module-presentation grain and do not interpret repeated appearances across presentations as duplicate people.

**Limitations:** These are cohort-size descriptions, not evidence of course quality or student outcomes. This is saved notebook evidence, not a fresh execution.

## AN2.04 — Gender, age band, and disability distribution

**Question:** How are gender, age band, and disability status distributed across presentations?

**Expectation:** Category counts and percentages should reconcile to each presentation denominator, with missing categories visible.

**Grain:** Student-presentation records grouped by demographic categories.

**Source:** student_info

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** These are descriptive distributions and do not establish causal relationships.

In [8]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN2.04")
query_results["AN2.04"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,dimension_name,category_value,category_rows,presentation_rows,pct_within_presentation
0,AAA,2013J,age_band,0-35,179,383,46.74
1,AAA,2013J,age_band,35-55,178,383,46.48
2,AAA,2013J,age_band,55<=,26,383,6.79
3,AAA,2014J,age_band,0-35,173,365,47.40
4,AAA,2014J,age_band,35-55,173,365,47.40
...,...,...,...,...,...,...,...
148,GGG,2013J,gender,M,178,952,18.70
149,GGG,2014B,gender,F,666,833,79.95
150,GGG,2014B,gender,M,167,833,20.05
151,GGG,2014J,gender,F,603,749,80.51


Query file: sql\04_analytics\AN2.04_gender_age_disability_distribution.sql
Elapsed seconds: 0.09
Displayed rows: 153; truncated: False


### Interpretation and decision — AN2.04

**Observed result:** The query returns 153 category-level rows across gender, age band, and disability status, with category counts and percentages relative to each presentation population.

**Decision:** Preserve recorded categories and presentation-specific denominators. Do not collapse categories or impute missing categories without a separately justified analytical rule.

**Limitations:** These profiles are descriptive and do not establish that demographic characteristics cause differences in learning or outcomes. Inspect the full result before citing a category-specific percentage.

**Output completeness note:** The query runner reports 153 returned rows and `truncated: False`, but the rendered Pandas representation abbreviates middle rows. Validate each dimension’s category totals and NULL/unclassified handling before detailed subgroup reporting.

## WO-1 reconciliation — AN2.04: Gender, age band, and disability

**Question:** Do category counts reconcile exactly to the relevant presentation denominator, retaining explicit NULL categories?

**Expectation:** Counts equal the denominator; the sum of rounded category percentages is approximately 100%; one denominator value exists per group.

**Grain:** One row per module presentation and dimension.

**Source:** Existing canonical read-only query `AN2.04` over `staging.student_info`.

**Safety:** READ-ONLY. No source data modification, imputation, or category collapsing.

**Runtime:** Low.

**Read as:** This verifies aggregation completeness and denominator consistency, not causal relationships. 153 rows across gender, age_band, and disability.


In [11]:
# 1. Run the existing canonical read-only query and reconcile category totals.
recon = run_query("AN2.04")
df = recon["data"].copy()
required = {"code_module", "code_presentation", "category_value", "category_rows", "presentation_rows", "pct_within_presentation", "dimension_name"}
assert required.issubset(df.columns), f"Unexpected columns: {list(df.columns)}"
assert not recon["display_truncated"], "Result was truncated by the query runner."

group_cols = ["code_module", "code_presentation", "dimension_name"]
check = (
    df.groupby(group_cols, dropna=False)
    .agg(
        category_total=("category_rows", "sum"),
        denominator=("presentation_rows", "first"),
        denominator_values=("presentation_rows", "nunique"),
        percentage_total=("pct_within_presentation", "sum"),
        null_category_present=("category_value", lambda s: s.eq("<NULL>").any()),
        categories=("category_value", "nunique"),
    )
    .reset_index()
)
check["counts_reconcile"] = check["category_total"].eq(check["denominator"]) & check["denominator_values"].eq(1)
check["percentages_reconcile"] = check["percentage_total"].between(99.90, 100.10)
display(check)
print(f"Query: {recon['query_file']} | elapsed: {recon['elapsed_seconds']} sec | rows: {recon['rows_displayed']}")
print("All counts reconcile:", bool(check["counts_reconcile"].all()))
print("All rounded percentages reconcile:", bool(check["percentages_reconcile"].all()))
print("Groups with explicit <NULL> category:", int(check["null_category_present"].sum()), "of", len(check))


,code_module,code_presentation,dimension_name,category_total,denominator,denominator_values,percentage_total,null_category_present,categories,counts_reconcile,percentages_reconcile
0,AAA,2013J,age_band,383,383,1,100.01,False,3,True,True
1,AAA,2013J,disability,383,383,1,100.00,False,2,True,True
2,AAA,2013J,gender,383,383,1,100.00,False,2,True,True
3,AAA,2014J,age_band,365,365,1,100.01,False,3,True,True
4,AAA,2014J,disability,365,365,1,100.00,False,2,True,True
...,...,...,...,...,...,...,...,...,...,...,...
61,GGG,2014B,disability,833,833,1,100.00,False,2,True,True
62,GGG,2014B,gender,833,833,1,100.00,False,2,True,True
63,GGG,2014J,age_band,749,749,1,100.00,False,3,True,True
64,GGG,2014J,disability,749,749,1,100.00,False,2,True,True


Query: sql\04_analytics\AN2.04_gender_age_disability_distribution.sql | elapsed: 0.082 sec | rows: 153
All counts reconcile: True
All rounded percentages reconcile: True
Groups with explicit <NULL> category: 0 of 66



### Interpretation and decision — AN2.04 demographic reconciliation

**Execution status:** Executed and validated.

**Observed result:**
- Reconciliation grain: module-presentation and demographic dimension.
- Dimensions checked: `age_band`, `disability`, and `gender`.
- Presentation-dimension groups checked: 66.
- Category counts reconciled with their presentation denominators: 66/66.
- Rounded percentage totals reconciled within the accepted tolerance: 66/66.
- Groups with an explicit `<NULL>` output category: 0/66.
- Underlying SQL output: 153 rows.
- Elapsed time: 0.082 seconds.

**Validation:**
Each presentation-dimension group contained exactly one denominator value. Category counts summed to the relevant presentation population, and rounded percentages reconciled within tolerance.

**Interpretation:**
The three demographic distributions are internally consistent at the tested presentation-level grain. Differences in category composition can be described, but they must not be interpreted as causal explanations of student outcomes.

**NULL limitation:**
No explicit `<NULL>` category was emitted in these grouped outputs. This does not establish source-wide completeness or explain the treatment of missing values in every source field.

**Decision:**
Accept AN2.04 count and percentage reconciliation for descriptive profiling, retaining the distinction between observed composition and causal interpretation.


## AN2.05 — Region and prior-education distribution

**Question:** How do region and prior-education categories vary across presentations?

**Expectation:** Category counts and percentages should reconcile to the stated denominator, with missing values retained explicitly.

**Grain:** Student-presentation records grouped by region and prior education.

**Source:** student_info

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Describe composition only; do not infer regional or educational causes of outcomes.

In [9]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN2.05")
query_results["AN2.05"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,dimension_name,category_value,category_rows,presentation_rows,pct_within_presentation
0,AAA,2013J,highest_education,A Level or Equivalent,194,383,50.65
1,AAA,2013J,highest_education,HE Qualification,98,383,25.59
2,AAA,2013J,highest_education,Lower Than A Level,83,383,21.67
3,AAA,2013J,highest_education,Post Graduate Qualification,8,383,2.09
4,AAA,2014J,highest_education,A Level or Equivalent,201,365,55.07
...,...,...,...,...,...,...,...
383,GGG,2014J,region,South Region,72,749,9.61
384,GGG,2014J,region,South West Region,62,749,8.28
385,GGG,2014J,region,Wales,51,749,6.81
386,GGG,2014J,region,West Midlands Region,51,749,6.81


Query file: sql\04_analytics\AN2.05_region_education_distribution.sql
Elapsed seconds: 0.055
Displayed rows: 388; truncated: False


### Interpretation and decision — AN2.05

**Observed result:** The query returns 388 category-level rows for region and highest education, with counts and percentages relative to each presentation population.

**Decision:** Retain source categories, including any explicit missing-value category, and use presentation-specific denominators. Avoid interpreting composition differences as causal effects.

**Limitations:** Do not publish category-specific claims without inspecting the relevant full rows. This is descriptive evidence only.

**Output completeness note:** The query runner reports 388 returned rows and `truncated: False`, but the rendered Pandas representation abbreviates middle rows. Validate each dimension’s category totals and NULL/unclassified handling before detailed subgroup reporting.

## WO-1 reconciliation — AN2.05: Region and highest education

**Question:** Do category counts reconcile exactly to the relevant presentation denominator, retaining explicit NULL categories?

**Expectation:** Counts equal the denominator; the sum of rounded category percentages is approximately 100%; one denominator value exists per group.

**Grain:** One row per module presentation and dimension.

**Source:** Existing canonical read-only query `AN2.05` over `staging.student_info`.

**Safety:** READ-ONLY. No source data modification, imputation, or category collapsing.

**Runtime:** Low.

**Read as:** This verifies aggregation completeness and denominator consistency, not causal relationships. 388 rows across region and highest_education.


In [12]:
# 1. Run the existing canonical read-only query and reconcile category totals.
recon = run_query("AN2.05")
df = recon["data"].copy()
required = {"code_module", "code_presentation", "category_value", "category_rows", "presentation_rows", "pct_within_presentation", "dimension_name"}
assert required.issubset(df.columns), f"Unexpected columns: {list(df.columns)}"
assert not recon["display_truncated"], "Result was truncated by the query runner."

group_cols = ["code_module", "code_presentation", "dimension_name"]
check = (
    df.groupby(group_cols, dropna=False)
    .agg(
        category_total=("category_rows", "sum"),
        denominator=("presentation_rows", "first"),
        denominator_values=("presentation_rows", "nunique"),
        percentage_total=("pct_within_presentation", "sum"),
        null_category_present=("category_value", lambda s: s.eq("<NULL>").any()),
        categories=("category_value", "nunique"),
    )
    .reset_index()
)
check["counts_reconcile"] = check["category_total"].eq(check["denominator"]) & check["denominator_values"].eq(1)
check["percentages_reconcile"] = check["percentage_total"].between(99.90, 100.10)
display(check)
print(f"Query: {recon['query_file']} | elapsed: {recon['elapsed_seconds']} sec | rows: {recon['rows_displayed']}")
print("All counts reconcile:", bool(check["counts_reconcile"].all()))
print("All rounded percentages reconcile:", bool(check["percentages_reconcile"].all()))
print("Groups with explicit <NULL> category:", int(check["null_category_present"].sum()), "of", len(check))


,code_module,code_presentation,dimension_name,category_total,denominator,denominator_values,percentage_total,null_category_present,categories,counts_reconcile,percentages_reconcile
0,AAA,2013J,highest_education,383,383,1,100.00,False,4,True,True
1,AAA,2013J,region,383,383,1,100.01,False,13,True,True
2,AAA,2014J,highest_education,365,365,1,100.00,False,3,True,True
3,AAA,2014J,region,365,365,1,100.01,False,13,True,True
4,BBB,2013B,highest_education,1767,1767,1,100.00,False,5,True,True
5,BBB,2013B,region,1767,1767,1,99.99,False,13,True,True
6,BBB,2013J,highest_education,2237,2237,1,99.99,False,5,True,True
7,BBB,2013J,region,2237,2237,1,100.00,False,13,True,True
8,BBB,2014B,highest_education,1613,1613,1,100.00,False,5,True,True
9,BBB,2014B,region,1613,1613,1,99.97,False,13,True,True


Query: sql\04_analytics\AN2.05_region_education_distribution.sql | elapsed: 0.061 sec | rows: 388
All counts reconcile: True
All rounded percentages reconcile: True
Groups with explicit <NULL> category: 0 of 44



### Interpretation and decision — AN2.05 region and education reconciliation

**Execution status:** Executed and validated.

**Observed result:**
- Reconciliation grain: module-presentation and profile dimension.
- Dimensions checked: `highest_education` and `region`.
- Presentation-dimension groups checked: 44.
- Category counts reconciled with their presentation denominators: 44/44.
- Rounded percentage totals reconciled within the accepted tolerance: 44/44.
- Groups with an explicit `<NULL>` output category: 0/44.
- Underlying SQL output: 388 rows.
- Elapsed time: 0.061 seconds.

**Validation:**
Each presentation-dimension group contained exactly one denominator value. Category counts summed to the corresponding presentation population, and rounded percentages reconciled within tolerance.

**Interpretation:**
The number of observed categories varies across presentations. This is not a reconciliation failure: the category totals still match their respective presentation denominators.

The results support descriptive comparisons of regional and educational composition. They do not establish that region or prior education causes a particular academic outcome.

**NULL limitation:**
No explicit `<NULL>` category was emitted in these grouped outputs. This must not be interpreted as proof that the source fields contain no SQL NULL values.

**Decision:**
Accept AN2.05 count and percentage reconciliation for descriptive profiling, retaining the presentation-specific category structure.


## AN2.06 — Numeric student profile

**Question:** What do the available numeric student-profile fields look like by presentation?

**Expectation:** Report valid counts and distribution summaries with missingness separated from observed numeric values.

**Grain:** Student-presentation records grouped by presentation.

**Source:** student_info

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Use summaries as descriptive context, not as validated predictive features.

In [13]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN2.06")
query_results["AN2.06"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,metric_name,total_rows,non_null_rows,null_rows,minimum_value,mean_value,median_value,maximum_value
0,AAA,2013J,num_of_prev_attempts,383,383,0,0,0.00,0.0,0
1,AAA,2014J,num_of_prev_attempts,365,365,0,0,0.11,0.0,1
2,BBB,2013B,num_of_prev_attempts,1767,1767,0,0,0.26,0.0,5
3,BBB,2013J,num_of_prev_attempts,2237,2237,0,0,0.21,0.0,5
4,BBB,2014B,num_of_prev_attempts,1613,1613,0,0,0.24,0.0,6
5,BBB,2014J,num_of_prev_attempts,2292,2292,0,0,0.16,0.0,5
6,CCC,2014B,num_of_prev_attempts,1936,1936,0,0,0.00,0.0,0
7,CCC,2014J,num_of_prev_attempts,2498,2498,0,0,0.09,0.0,1
8,DDD,2013B,num_of_prev_attempts,1303,1303,0,0,0.28,0.0,6
9,DDD,2013J,num_of_prev_attempts,1938,1938,0,0,0.20,0.0,5


Query file: sql\04_analytics\AN2.06_numeric_student_profile.sql
Elapsed seconds: 0.189
Displayed rows: 44; truncated: False


### Interpretation and decision — AN2.06

**Observed result:** The query returns 44 presentation-by-metric rows for `num_of_prev_attempts` and `studied_credits`. Both fields have zero NULLs in the saved output. Median previous attempts is zero across presentations, while the maximum reaches 6. Mean studied credits range from 33.38 to 92.66; the maximum observed value is 655.

**Decision:** Report robust summaries alongside means. A median of zero does not mean every student has zero previous attempts, and large values should be investigated rather than automatically discarded.

**Limitations:** These summaries do not determine whether an extreme credit value is erroneous or educationally invalid.

## AN2.07 — IMD missingness by presentation

**Question:** How does missing IMD band information vary by presentation?

**Expectation:** Missing and non-missing counts should sum to the presentation population, and the missing rate should use that population as denominator.

**Grain:** Student-presentation records grouped by presentation.

**Source:** student_info

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** A missingness rate describes data availability, not deprivation level.

In [14]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN2.07")
query_results["AN2.07"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,presentation_rows,imd_missing_rows,imd_present_rows,pct_imd_missing
0,FFF,2013B,1614,95,1519,5.89
1,CCC,2014J,2498,146,2352,5.84
2,CCC,2014B,1936,108,1828,5.58
3,EEE,2013J,1052,52,1000,4.94
4,FFF,2014B,1500,72,1428,4.80
5,DDD,2014B,1228,58,1170,4.72
6,DDD,2013B,1303,61,1242,4.68
7,DDD,2014J,1803,81,1722,4.49
8,FFF,2013J,2283,101,2182,4.42
9,FFF,2014J,2365,99,2266,4.19


Query file: sql\04_analytics\AN2.07_imd_missing_by_presentation.sql
Elapsed seconds: 0.034
Displayed rows: 22; truncated: False


### Interpretation and decision — AN2.07

**Observed result:** The saved output reports 1,111 missing `imd_band` values across 32,593 student-presentation rows. The highest missingness rate is 5.89% in FFF 2013B (95/1,614); the lowest is 0.13% in GGG 2014J (1/749).

**Decision:** Preserve missing IMD values and report counts with presentation denominators. Do not treat missingness as a socioeconomic category or silently impute values at this stage.

**Limitations:** Differences in missingness are descriptive and do not establish the mechanism or cause of missing data.

## AN2.08 — IMD missingness by region

**Question:** Does IMD band missingness vary across region categories?

**Expectation:** Report missing and non-missing counts with the denominator for each region category clearly defined.

**Grain:** Student-presentation records grouped by region.

**Source:** student_info

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Do not interpret missing IMD values as low or high deprivation.

In [15]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN2.08")
query_results["AN2.08"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,region_category,student_presentation_rows,imd_missing_rows,imd_present_rows,pct_imd_missing
0,North Region,1823,731,1092,40.10
1,Ireland,1184,266,918,22.47
2,South Region,3092,48,3044,1.55
3,West Midlands Region,2582,39,2543,1.51
4,Scotland,3446,12,3434,0.35
5,Yorkshire Region,2006,5,2001,0.25
6,South West Region,2436,5,2431,0.21
7,North Western Region,2906,5,2901,0.17
8,East Anglian Region,3340,0,3340,0.00
9,London Region,3216,0,3216,0.00


Query file: sql\04_analytics\AN2.08_imd_missing_by_region.sql
Elapsed seconds: 0.023
Displayed rows: 13; truncated: False


### Interpretation and decision — AN2.08

**Observed result:** The region profile contains 13 categories. North Region has 731 missing IMD values among 1,823 student-presentation rows (40.10%); Ireland has 266/1,184 (22.47%). Five categories have zero missing IMD values: East Anglian Region, London Region, East Midlands Region, South East Region, and Wales.

**Decision:** Preserve region-specific denominators and investigate the high missingness in North Region and Ireland before using IMD in subgroup comparisons. Report zero-missing categories as observed, not as proof of overall data quality.

**Limitations:** These patterns are descriptive and do not establish why IMD is missing.

## AN2.09 — IMD missingness by final result

**Question:** Does IMD band missingness differ across final-result categories?

**Expectation:** Missing and non-missing counts and rates should be reported for each outcome category, including missing outcomes if represented.

**Grain:** Student-presentation records grouped by final result.

**Source:** student_info

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Do not infer that IMD missingness causes a final result.

In [16]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN2.09")
query_results["AN2.09"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,final_result_category,student_presentation_rows,imd_missing_rows,imd_present_rows,pct_imd_missing
0,Distinction,3024,199,2825,6.58
1,Pass,12361,531,11830,4.30
2,Withdrawn,10156,236,9920,2.32
3,Fail,7052,145,6907,2.06


Query file: sql\04_analytics\AN2.09_imd_missing_by_final_result.sql
Elapsed seconds: 0.017
Displayed rows: 4; truncated: False


### Interpretation and decision — AN2.09

**Observed result:** The output totals 32,593 student-presentation rows and 1,111 missing IMD values. Missingness is 6.58% for Distinction (199/3,024), 4.30% for Pass (531/12,361), 2.32% for Withdrawn (236/10,156), and 2.06% for Fail (145/7,052).

**Decision:** Keep final-result categories and denominators explicit. Do not impute or exclude missing IMD values based on final result without a documented rationale.

**Limitations:** These are descriptive associations, not evidence that final result causes IMD missingness or vice versa. This is saved notebook evidence, not a fresh execution.


## WO-1 reconciliation gate — final interpretation

**Execution status:** All three targeted reconciliation checks have been executed and validated.

| Query | Groups checked | Count reconciliation | Rounded percentage reconciliation | Explicit `<NULL>` groups |
|---|---:|---:|---:|---:|
| AN2.02 | 22 | 22/22 PASS | 22/22 PASS | 0 |
| AN2.04 | 66 | 66/66 PASS | 66/66 PASS | 0 |
| AN2.05 | 44 | 44/44 PASS | 44/44 PASS | 0 |
| **Total** | **132** | **132/132 PASS** | **132/132 PASS** | **0/132** |

**Conclusion:**

All 132 presentation/dimension groups passed category-count reconciliation and rounded-percentage reconciliation. Each group used one consistent denominator.

Small percentage deviations around 100% are attributable to rounding of displayed category percentages.

The absence of an explicit `<NULL>` output category does not establish source-wide absence of SQL NULL values. Previously documented missingness, including the 1,111 missing `imd_band` values, remains relevant.

**Work-order decision:**

The targeted reconciliation requirement for AN2.02, AN2.04, and AN2.05 is complete. Together with the previously reviewed AN0–AN2 execution evidence, this closes the defined WO-1 analytical-readiness scope.

The remaining semantic questions are carried into the relevant downstream analytical stages. Completed diagnostics must not be repeated without a specific evidence-based reason.

**Next analytical stages:**

- AN3 — Outcomes and withdrawal.
- AN4 — Assessment analysis.
- AN5 — Engagement sensitivity, including R0/R1/R2, after defining the engagement question and investigating repeated `student_vle` records.

No KPI, causal claim, dashboard finding, or predictive result is approved by this readiness closure.


## WO-1 acceptance criteria and current disposition

Saved execution outputs are retained for AN0.01–AN0.04, AN1.01–AN1.09, and AN2.01–AN2.09. Their interpretations and limitations are recorded in `docs/analysis_log.md`. The execution evidence comes from a prior local PostgreSQL session and is not a fresh execution during this review.

The targeted reconciliation for AN2.02, AN2.04, and AN2.05 has been executed and saved. All 132 presentation/dimension groups passed category-count and rounded-percentage reconciliation. The absence of an explicit `<NULL>` output category does not establish source-wide absence of SQL NULL values. WO-1 does not itself create analytical views, validated KPIs, dashboards, predictive models, or business recommendations.

Only aggregate query outputs and relevant error details are retained in the analytical record. Environment files, passwords, private connection strings, and raw CSV files are excluded.